# ZARX-1B Training Notebook (Google Colab)
1B parameter code + ARC reasoning model trained from scratch.

**GPU:** T4 (16GB VRAM) | **Runtime:** 4-12hrs per session

## Persistence
- Google Drive: saves data + checkpoints (survives disconnects)
- GitHub: pushes checkpoints to codedbytahir/zarx-checkpoints
- HuggingFace: cross-platform checkpoint sync

## Setup
1. Go to Runtime -> Change runtime type -> T4 GPU
2. Run cells in order
3. Training auto-resumes from checkpoints

In [ ]:
#@title 1. Install Dependencies
!pip install -q torch>=2.1.0 bitsandbytes wandb huggingface_hub
!pip install -q datasets tokenizers accelerate datasketch
# Flash Attention skipped on T4 (not supported, SDPA auto-fallback works)
import torch
if torch.cuda.is_available() and ('A100' in torch.cuda.get_device_name() or 'L4' in torch.cuda.get_device_name()):
    !pip install -q flash-attn --no-build-isolation
print('Dependencies installed!')

In [ ]:
#@title 2. Mount Google Drive + Login
from google.colab import drive
drive.mount('/content/drive')

import os

# Create persistence directories on Drive
DRIVE_DIR = '/content/drive/MyDrive/ZARX-1B'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data/processed', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/tokenizer', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/configs', exist_ok=True)

# HuggingFace login
HF_TOKEN = '' #@param {type:"string"}
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

# W&B login
WANDB_KEY = '' #@param {type:"string"}
if WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)

# GitHub token for checkpoint pushes
GH_TOKEN = '' #@param {type:"string"}

print('Drive mounted and logins complete!')

In [ ]:
#@title 3. Clone ZARX-1B Repository
%cd /content
!rm -rf ZARX-1B  # Clean if re-running
!git clone https://github.com/codedbytahir/ZARX-1B.git
%cd ZARX-1B
print('Repository cloned!')

In [ ]:
#@title 4. Restore Data from Drive (if exists) OR Download Fresh
import shutil
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/ZARX-1B')
DATA = Path('/content/data')

# Try restore from Drive first
restored = False
if (DRIVE / 'data' / 'processed').exists() and any((DRIVE / 'data' / 'processed').glob('*.jsonl')):
    print('Found processed data on Drive! Restoring...')
    shutil.copytree(str(DRIVE / 'data' / 'processed'), str(DATA / 'processed'), dirs_exist_ok=True)
    restored = True
    print('Data restored from Drive!')

# Restore tokenizer if on Drive
if (DRIVE / 'tokenizer' / 'tokenizer.json').exists():
    shutil.copy2(str(DRIVE / 'tokenizer' / 'tokenizer.json'), 'configs/tokenizer.json')
    print('Tokenizer restored from Drive!')

if not restored:
    print('No data on Drive. Need to run full setup:')
    print('  !python scripts/setup_colab_day1.py --hf_token YOUR_TOKEN --wandb_key YOUR_KEY --github_token YOUR_GH_TOKEN')

In [ ]:
#@title 5. Verify GPU & Environment
import torch
import sys
sys.path.insert(0, '.')

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'BF16 supported: {torch.cuda.is_bf16_supported()}')

# Quick model test
from src.model import build_model
model = build_model('configs/model_config.json')
model = model.cuda()
x = torch.randint(0, 32000, (1, 128)).cuda()
out = model(x)
print(f'Forward pass OK! Output shape: {out["logits"].shape}')
print(f'GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB used')
del model, x, out
torch.cuda.empty_cache()

In [ ]:
#@title 6. Start Training (auto-resumes, saves to Drive + GitHub + HF)
HF_REPO = '' #@param {type:"string"}  # Your HuggingFace checkpoint repo

!python src/train.py \
    --model_config configs/model_config.json \
    --tokenizer_path configs/tokenizer.json \
    --data_path /content/data/processed \
    --micro_batch_size 1 \
    --gradient_accumulation_steps 32 \
    --learning_rate 3e-4 \
    --warmup_steps 2000 \
    --total_tokens 10000000000 \
    --use_8bit_adam \
    --checkpoint_dir /content/checkpoints \
    --drive_dir /content/drive/MyDrive/ZARX-1B \
    --github_token {GH_TOKEN} \
    --github_repo codedbytahir/zarx-checkpoints \
    --hf_repo_id {HF_REPO} \
    --hf_token {HF_TOKEN} \
    --save_every_local 100 \
    --save_every_drive 500 \
    --save_every_github 1000 \
    --save_every_hf 1000 \
    --log_every_steps 10 \
    --wandb_project zarx-1b \
    --output_dir /content/zarx-1b-final

In [ ]:
#@title 7. Emergency Save - Push Everything to Drive + GitHub
# Run this cell if you think the session is about to die!
import shutil
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/ZARX-1B')

# Save checkpoints
if Path('/content/checkpoints').exists():
    shutil.copytree('/content/checkpoints', str(DRIVE / 'checkpoints'), dirs_exist_ok=True)
    print('Checkpoints saved to Drive!')

# Save data
if Path('/content/data/processed').exists():
    shutil.copytree('/content/data/processed', str(DRIVE / 'data' / 'processed'), dirs_exist_ok=True)
    print('Processed data saved to Drive!')

# Save tokenizer
if Path('configs/tokenizer.json').exists():
    shutil.copy2('configs/tokenizer.json', str(DRIVE / 'tokenizer' / 'tokenizer.json'))
    print('Tokenizer saved to Drive!')

# Push to GitHub
if GH_TOKEN:
    !cd /content/zarx-checkpoints && git add -A && git commit -m 'emergency checkpoint save' && git push https://{GH_TOKEN}@github.com/codedbytahir/zarx-checkpoints.git main
    print('Pushed to GitHub!')

print('EMERGENCY SAVE COMPLETE! Everything is persisted.')